# 4. K-Nearest Neighbors (KNN)

**Machine Learning Fundamentals and Predictive Analytics — Notebook 4 of 11**

KNN is the simplest idea in machine learning: **to predict a new point, look at the $k$
training points nearest to it and copy them.** Majority vote for classification, average for
regression.

There is no training. The "model" is the training set. This makes KNN a **lazy learner** — it
does all its work at prediction time — and an excellent vehicle for learning about distance
metrics, feature scaling, and the curse of dimensionality.

### What you will learn

1. The algorithm, and what "no training" really means
2. **Distance metrics**: Euclidean, Manhattan, Minkowski, cosine
3. Why **feature scaling is not optional** for KNN
4. Choosing $k$, and how $k$ controls the bias-variance trade-off
5. **Distance weighting** and tie-breaking
6. **KNN regression**
7. The **curse of dimensionality**, demonstrated
8. Computational cost, and the tree structures that reduce it
9. Handling imbalance and categorical features
10. When KNN is a good idea, and when it is not

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import (KNeighborsClassifier, KNeighborsRegressor, NearestNeighbors)
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     StratifiedKFold, KFold, validation_curve)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_squared_error, r2_score)
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier

rng = np.random.default_rng(seed=4)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)
CV = KFold(5, shuffle=True, random_state=0)

---
## 4.1 The algorithm

To predict for a new point $\mathbf{x}$:

1. Compute the distance from $\mathbf{x}$ to **every** training point
2. Keep the $k$ closest
3. **Classification:** predict the majority class among them.
   **Regression:** predict their mean.
4. Predicted probability = the fraction of the $k$ neighbours in each class

That is the entire algorithm. `fit()` in scikit-learn merely stores the data (and builds an
index to make step 1 faster).

Three consequences worth stating up front:

- **Nothing is learned**, so there are no coefficients to interpret and no model to inspect
- **Prediction is expensive** and scales with the size of the training set
- The **notion of distance is the model**. Everything about KNN's behaviour follows from how
  you measure distance — which is why scaling matters so much.

In [ ]:
# KNN from scratch, so there is no mystery
def knn_predict(X_train, y_train, x_new, k=5):
    '''Majority-vote KNN classification for one query point.'''
    distances = np.sqrt(((X_train - x_new) ** 2).sum(axis=1))
    nearest = np.argsort(distances)[:k]
    votes = np.bincount(y_train[nearest])
    return int(np.argmax(votes)), nearest, distances[nearest]

# A tiny 2-D dataset
Xt = np.array([[1, 1], [1.5, 2], [2, 1.2], [3, 3.4], [3.5, 2.8], [4, 3.6],
               [1.2, 3.4], [0.8, 2.9], [4.2, 1.1], [3.8, 0.6]])
yt = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 1])
query = np.array([2.6, 2.4])

pred, nbrs, dists = knn_predict(Xt, yt, query, k=3)
print(f"Query point {query}")
print(f"3 nearest neighbours: indices {nbrs.tolist()}, classes {yt[nbrs].tolist()}")
print(f"  distances: {np.round(dists, 3).tolist()}")
print(f"Majority vote -> predicted class {pred}")

sk = KNeighborsClassifier(n_neighbors=3).fit(Xt, yt)
print(f"\nscikit-learn agrees: {sk.predict([query])[0]}, "
      f"probabilities {sk.predict_proba([query])[0]}")

In [ ]:
plt.scatter(Xt[yt == 0, 0], Xt[yt == 0, 1], s=110, color="steelblue", label="class 0",
            edgecolor="k")
plt.scatter(Xt[yt == 1, 0], Xt[yt == 1, 1], s=110, color="crimson", label="class 1",
            edgecolor="k")
plt.scatter(*query, s=250, marker="*", color="gold", edgecolor="k", zorder=5,
            label="query point")
radius = dists.max()
plt.gca().add_patch(plt.Circle(query, radius, fill=False, ls="--", color="black"))
for i in nbrs:
    plt.plot([query[0], Xt[i, 0]], [query[1], Xt[i, 1]], color="grey", lw=1.2, zorder=1)
plt.gca().set_aspect("equal")
plt.title("k = 3: the circle contains exactly the three nearest points")
plt.legend(fontsize=8); plt.show()

print("Changing k changes the radius of that circle, and therefore the vote.")
for k in (1, 3, 5, 7, 9):
    p, _, _ = knn_predict(Xt, yt, query, k=k)
    print(f"  k = {k}: predicted class {p}")

---
## 4.2 Distance metrics

The **Minkowski distance** of order $p$ generalises the familiar cases:

$$d(\mathbf{a}, \mathbf{b}) = \left(\sum_{j=1}^{d}|a_j - b_j|^p\right)^{1/p}$$

| $p$ | Name | Character |
|---|---|---|
| 1 | **Manhattan** (city block) | Sum of absolute differences; robust to a single large gap |
| 2 | **Euclidean** | Straight line; the default |
| $\infty$ | **Chebyshev** | The single largest coordinate difference |

Also useful:

- **Cosine distance** $1 - \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ —
  ignores magnitude, compares *direction*. The right choice for text (TF-IDF vectors) and
  recommender systems.
- **Hamming** — for binary or categorical features: the fraction of positions that differ.

The metric encodes an assumption about what "similar" means. Choose it deliberately.

In [ ]:
a, b = np.array([1.0, 2.0, 3.0]), np.array([4.0, 0.0, 3.5])

print(f"a = {a}, b = {b}\n")
print(f"Manhattan (p=1) : {np.abs(a-b).sum():.4f}")
print(f"Euclidean (p=2) : {np.sqrt(((a-b)**2).sum()):.4f}")
print(f"Minkowski (p=3) : {(np.abs(a-b)**3).sum()**(1/3):.4f}")
print(f"Chebyshev       : {np.abs(a-b).max():.4f}")
print(f"Cosine distance : {1 - a@b/(np.linalg.norm(a)*np.linalg.norm(b)):.4f}")

# Unit "circles" for different metrics: the shape of what counts as 'nearby'
theta = np.linspace(0, 2*np.pi, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (p, name) in zip(axes, [(1, "Manhattan (p=1)"), (2, "Euclidean (p=2)"),
                                (8, "approaching Chebyshev (p=8)")]):
    ang = np.linspace(0, 2*np.pi, 800)
    r = 1 / (np.abs(np.cos(ang))**p + np.abs(np.sin(ang))**p) ** (1/p)
    ax.plot(r*np.cos(ang), r*np.sin(ang), color="steelblue", lw=2.4)
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_title(f"{name}\nall points at distance 1", fontsize=9)
plt.tight_layout(); plt.show()
print("The 'unit circle' is a diamond for p=1, a circle for p=2, and approaches a square")
print("as p grows. That shape IS the model's idea of similarity.")

In [ ]:
# Cosine vs Euclidean: when magnitude should be ignored
docs = pd.DataFrame([[2, 1, 0], [20, 10, 0], [0, 1, 2]],
                    index=["short doc A", "long doc A (same topic)", "doc B (other topic)"],
                    columns=["word_ml", "word_data", "word_cooking"])
print(docs, "\n")

from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
print("Euclidean distances:")
print(pd.DataFrame(euclidean_distances(docs).round(2), index=docs.index, columns=docs.index))
print("\nCosine distances:")
print(pd.DataFrame(cosine_distances(docs).round(3), index=docs.index, columns=docs.index))
print("\nBy Euclidean distance the two documents on the SAME topic are far apart, purely")
print("because one is longer. Cosine distance sees that they point the same way.")
print("This is why text similarity always uses cosine.")

---
## 4.3 Feature scaling is mandatory

KNN adds up squared differences across features. If one feature is measured in rupees
(0–1,000,000) and another in years (0–50), the rupees term dominates completely and the years
feature is effectively ignored.

**Unscaled KNN silently uses only your largest-range feature.** This is the single most common
KNN bug.

| Scaler | Formula | Use when |
|---|---|---|
| `StandardScaler` | $(x - \mu)/\sigma$ | Roughly Normal features (the default choice) |
| `MinMaxScaler` | $(x - \min)/(\max - \min)$ | Bounded features; you want [0,1] |
| `RobustScaler` | $(x - \text{median})/\text{IQR}$ | Heavy outliers |

Always inside a `Pipeline`, so the scaler is fitted on the training fold only.

In [ ]:
# Two features with genuinely equal predictive value but wildly different scales
m = 800
income = rng.normal(600_000, 150_000, m)         # range ~ hundreds of thousands
years_edu = rng.normal(14, 3, m)                 # range ~ tens
# The label depends EQUALLY on the standardised versions of both
z = 1.2*(income - income.mean())/income.std() + 1.2*(years_edu - years_edu.mean())/years_edu.std()
label = (z + rng.normal(0, 0.7, m) > 0).astype(int)

Xs_ = pd.DataFrame({"income": income, "years_edu": years_edu})
Xa, Xb, ya, yb = train_test_split(Xs_, label, test_size=0.3, random_state=0, stratify=label)

unscaled = KNeighborsClassifier(n_neighbors=15).fit(Xa, ya)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)).fit(Xa, ya)

print(f"KNN without scaling : test accuracy {unscaled.score(Xb, yb):.4f}")
print(f"KNN with scaling    : test accuracy {scaled.score(Xb, yb):.4f}")
print(f"Baseline            : "
      f"{DummyClassifier(strategy='most_frequent').fit(Xa, ya).score(Xb, yb):.4f}")

# Proof that the unscaled model is ignoring years_edu: drop it and nothing changes
only_income = KNeighborsClassifier(n_neighbors=15).fit(Xa[["income"]], ya)
print(f"\nUnscaled KNN using ONLY income: {only_income.score(Xb[['income']], yb):.4f}")
print("Essentially identical to the unscaled two-feature model -- the education column")
print("contributed nothing, because a 3-year difference is invisible next to a 150,000")
print("difference in income.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].scatter(Xa.income, Xa.years_edu, c=ya, cmap="coolwarm", s=18, edgecolor="k",
                linewidth=0.2)
axes[0].set_xlabel("income"); axes[0].set_ylabel("years of education")
axes[0].set_title("Raw scale: the y-axis is compressed to nothing")

Xa_sc = pd.DataFrame(StandardScaler().fit_transform(Xa), columns=Xa.columns)
axes[1].scatter(Xa_sc.income, Xa_sc.years_edu, c=ya, cmap="coolwarm", s=18, edgecolor="k",
                linewidth=0.2)
axes[1].set_xlabel("income (standardised)"); axes[1].set_ylabel("years of education (standardised)")
axes[1].set_title("Standardised: both features can contribute")
axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()

print("Compare the three scalers:")
for name, sc in [("StandardScaler", StandardScaler()), ("MinMaxScaler", MinMaxScaler())]:
    acc = cross_val_score(make_pipeline(sc, KNeighborsClassifier(15)), Xs_, label,
                          cv=SKF).mean()
    print(f"  {name:<16} CV accuracy {acc:.4f}")
print(f"  {'no scaling':<16} CV accuracy "
      f"{cross_val_score(KNeighborsClassifier(15), Xs_, label, cv=SKF).mean():.4f}")

---
## 4.4 Choosing k

$k$ is the only important hyperparameter, and it controls capacity **inversely**:

| $k$ | Capacity | Behaviour |
|---|---|---|
| $k = 1$ | maximum | Each point is its own neighbour → **training error is 0**, boundary is jagged, high variance |
| moderate $k$ | balanced | Smooth boundary that follows the class structure |
| $k = n$ | minimum | Every prediction is the global majority → maximum bias |

Practical guidance:

- Tune by cross-validation. Do not guess.
- Use an **odd** $k$ for two classes to avoid ties.
- $k \approx \sqrt{n}$ is a reasonable starting point, not a rule.
- If the best $k$ is 1, be suspicious: it often signals duplicate rows or leakage.

In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=400, noise=0.28, random_state=1)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0,
                                              stratify=ym)

def plot_knn_boundary(k, ax, weights="uniform"):
    mdl = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(n_neighbors=k, weights=weights)).fit(Xm_tr, ym_tr)
    h = 0.02
    xx, yy = np.meshgrid(np.arange(Xm[:, 0].min()-0.5, Xm[:, 0].max()+0.5, h),
                         np.arange(Xm[:, 1].min()-0.5, Xm[:, 1].max()+0.5, h))
    Z = mdl.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=ym_tr, cmap="coolwarm", s=16,
               edgecolor="k", linewidth=0.2)
    ax.set_title(f"k = {k}\ntrain {mdl.score(Xm_tr, ym_tr):.3f} / test {mdl.score(Xm_te, ym_te):.3f}",
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, k in zip(axes, [1, 5, 15, 50, 200]):
    plot_knn_boundary(k, ax)
plt.tight_layout(); plt.show()

print("k=1   : perfect training score, islands of noise -> high variance")
print("k=15  : smooth boundary that follows the two moons")
print("k=200 : the boundary has almost vanished -> high bias")

In [ ]:
ks = np.arange(1, 81, 2)
tr_s, va_s = validation_curve(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                              Xm, ym, param_name="kneighborsclassifier__n_neighbors",
                              param_range=ks, cv=SKF, scoring="accuracy")
tr_m, va_m, va_sd = tr_s.mean(1), va_s.mean(1), va_s.std(1)

plt.plot(ks, tr_m, "o-", color="steelblue", ms=4, label="training accuracy")
plt.plot(ks, va_m, "o-", color="crimson", ms=4, label="cross-validated accuracy")
plt.fill_between(ks, va_m-va_sd, va_m+va_sd, color="crimson", alpha=0.15)
best_k = ks[int(np.argmax(va_m))]
plt.axvline(best_k, color="black", ls="--", label=f"best k = {best_k}")
plt.axvline(np.sqrt(len(ym)), color="grey", ls=":", label=f"sqrt(n) = {np.sqrt(len(ym)):.0f}")
plt.xlabel("k"); plt.ylabel("accuracy")
plt.title("Validation curve for k (note: capacity DECREASES to the right)")
plt.legend(fontsize=8); plt.show()

print(f"k=1 training accuracy : {tr_m[0]:.4f}  <- always exactly 1.0")
print(f"Best k by CV          : {best_k} (accuracy {va_m.max():.4f})")
thresh = va_m.max() - va_sd[int(np.argmax(va_m))]
print(f"One-standard-error rule: largest k within 1 SE is "
      f"{ks[np.max(np.where(va_m >= thresh))]}")

### Distance weighting

With `weights="distance"`, closer neighbours get more say — each vote is weighted by
$1/d$. This softens the choice of $k$: distant neighbours are included but count for little,
so performance degrades more gracefully as $k$ grows.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (k, w) in zip(axes, [(5, "uniform"), (5, "distance"),
                             (50, "uniform"), (50, "distance")]):
    plot_knn_boundary(k, ax, weights=w)
    ax.set_title(ax.get_title() + f"\nweights = {w}", fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'k':>5}{'uniform':>12}{'distance':>12}")
for k in (1, 5, 15, 50, 150, 300):
    u = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                        Xm, ym, cv=SKF).mean()
    d = cross_val_score(make_pipeline(StandardScaler(),
                                      KNeighborsClassifier(k, weights="distance")),
                        Xm, ym, cv=SKF).mean()
    print(f"{k:>5}{u:>12.4f}{d:>12.4f}")
print("\nDistance weighting keeps large k usable -- useful when you do not want to tune")
print("k precisely. Note that with weights='distance', k=1 and large k differ far less.")

---
## 4.5 KNN regression

Identical algorithm, but the prediction is the **average** (or distance-weighted average) of
the neighbours' target values.

Like a decision tree, KNN regression is **piecewise constant** with uniform weights, and it
**cannot extrapolate** — beyond the data it returns the average of the same $k$ boundary
points forever.

In [ ]:
xs = np.sort(rng.uniform(0, 10, 120))
ys = np.sin(xs) + 0.25*xs + rng.normal(0, 0.3, 120)
Xr = xs.reshape(-1, 1)
grid = np.linspace(-2, 14, 700).reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (k, w) in zip(axes, [(1, "uniform"), (5, "uniform"), (5, "distance"), (40, "uniform")]):
    kr = KNeighborsRegressor(n_neighbors=k, weights=w).fit(Xr, ys)
    ax.scatter(xs, ys, s=14, alpha=0.6, color="steelblue")
    ax.plot(grid, kr.predict(grid), color="crimson", lw=2)
    ax.axvspan(-2, 0, color="grey", alpha=0.12)
    ax.axvspan(10, 14, color="grey", alpha=0.12)
    ax.set_title(f"k={k}, weights={w}\ntrain R^2 {kr.score(Xr, ys):.3f}", fontsize=9)
plt.tight_layout(); plt.show()

print("Grey bands are outside the training range.")
kr5 = KNeighborsRegressor(5).fit(Xr, ys)
print(f"Prediction at x=11 : {kr5.predict([[11]])[0]:.3f}")
print(f"Prediction at x=50 : {kr5.predict([[50]])[0]:.3f}  <- identical, forever")
print("\nKNN interpolates well and extrapolates not at all.")

In [ ]:
# KNN regression on real data, tuned properly
from sklearn.datasets import load_diabetes
dia = load_diabetes()
Xd, yd = dia.data, dia.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)

gs = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                  {"kneighborsregressor__n_neighbors": range(1, 51, 2),
                   "kneighborsregressor__weights": ["uniform", "distance"],
                   "kneighborsregressor__p": [1, 2]},
                  cv=CV, scoring="neg_root_mean_squared_error").fit(Xd_tr, yd_tr)

print(f"Best parameters : {gs.best_params_}")
print(f"Best CV RMSE    : {-gs.best_score_:.3f}")
print(f"Test RMSE       : {np.sqrt(mean_squared_error(yd_te, gs.predict(Xd_te))):.3f}")
print(f"Test R^2        : {r2_score(yd_te, gs.predict(Xd_te)):.4f}")

from sklearn.linear_model import LinearRegression
lin = make_pipeline(StandardScaler(), LinearRegression()).fit(Xd_tr, yd_tr)
print(f"\nLinear regression test RMSE : "
      f"{np.sqrt(mean_squared_error(yd_te, lin.predict(Xd_te))):.3f}")
print("On this dataset the relationship is close to linear and the dimension is 10,")
print("so linear regression is competitive with (or better than) KNN. That is typical:")
print("KNN shines on low-dimensional data with genuinely local structure.")

---
## 4.6 The curse of dimensionality

This is the reason KNN fails on wide data, and it is worth understanding properly.

In high dimensions:

1. **Everything is far away.** The volume of the space grows exponentially, so a fixed number
   of points becomes hopelessly sparse.
2. **All distances become similar.** The ratio between the nearest and farthest neighbour
   approaches 1, so "nearest" stops being meaningful.
3. **Neighbourhoods are not local.** To capture even 1% of the volume in 50 dimensions, a
   hypercube must span 91% of the range of every feature.

Let's measure all three.

In [ ]:
# (1) and (2): distance concentration
print(f"{'dimensions':>12}{'mean dist':>12}{'nearest':>10}{'farthest':>11}{'far/near':>10}")
for d in (1, 2, 5, 10, 50, 200, 1000):
    P = rng.random((500, d))
    q = rng.random(d)
    dist = np.sqrt(((P - q) ** 2).sum(axis=1))
    print(f"{d:>12}{dist.mean():>12.3f}{dist.min():>10.3f}{dist.max():>11.3f}"
          f"{dist.max()/dist.min():>10.3f}")
print("\nIn 1-D the farthest point is ~50x further than the nearest. In 1000-D it is 1.2x.")
print("When every point is roughly equidistant, 'the k nearest' is close to a random sample.")

In [ ]:
# (3) how wide must a neighbourhood be to contain a fixed fraction of the volume?
fracs = [0.001, 0.01, 0.1]
dims = np.arange(1, 51)
for f_ in fracs:
    plt.plot(dims, f_ ** (1/dims), lw=2, label=f"{f_:.1%} of the volume")
plt.axhline(1.0, color="black", ls=":")
plt.xlabel("number of dimensions"); plt.ylabel("required edge length (fraction of range)")
plt.title("To stay 'local' in high dimensions you must span nearly the whole space")
plt.legend(fontsize=8); plt.show()

print(f"{'dimensions':>12}{'edge for 1% of volume':>24}")
for d in (1, 2, 5, 10, 20, 50):
    print(f"{d:>12}{0.01 ** (1/d):>24.3f}")
print("\nIn 50 dimensions, a 'neighbourhood' holding 1% of the data covers 91% of the range")
print("of every single feature. It is not a neighbourhood in any useful sense.")

In [ ]:
# The practical consequence: KNN accuracy collapses as you add noise dimensions
def knn_vs_dimension(n_noise, m_=600, k=15):
    signal = rng.normal(size=(m_, 2))
    y_ = ((signal[:, 0] + signal[:, 1]) > 0).astype(int)
    if n_noise:
        Xfull = np.column_stack([signal, rng.normal(size=(m_, n_noise))])
    else:
        Xfull = signal
    knn_acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                              Xfull, y_, cv=SKF).mean()
    from sklearn.linear_model import LogisticRegression
    lr_acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                             Xfull, y_, cv=SKF).mean()
    return knn_acc, lr_acc

noise_counts = [0, 2, 5, 10, 25, 50, 100, 200]
knn_scores, lr_scores = [], []
for nn in noise_counts:
    a_, b_ = knn_vs_dimension(nn)
    knn_scores.append(a_); lr_scores.append(b_)

plt.plot(noise_counts, knn_scores, "o-", color="steelblue", label="KNN (k=15)")
plt.plot(noise_counts, lr_scores, "o-", color="seagreen", label="logistic regression")
plt.xlabel("number of added NOISE features"); plt.ylabel("CV accuracy")
plt.title("KNN degrades badly with irrelevant features; a linear model shrugs them off")
plt.legend(fontsize=8); plt.show()

print(f"{'noise features':>16}{'KNN':>10}{'logistic':>11}")
for nn, a_, b_ in zip(noise_counts, knn_scores, lr_scores):
    print(f"{nn:>16}{a_:>10.4f}{b_:>11.4f}")
print("\nThe signal is unchanged throughout -- only irrelevant columns were added.")
print("Every noise dimension adds to the distance, drowning the two that matter.")
print("\nRemedies: feature selection, dimensionality reduction (PCA), or metric learning.")

In [ ]:
# PCA before KNN as a remedy
signal = rng.normal(size=(600, 2))
y_pca = ((signal[:, 0] + signal[:, 1]) > 0).astype(int)
X_wide = np.column_stack([signal, rng.normal(size=(600, 100))])

plain = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
                        X_wide, y_pca, cv=SKF).mean()
print(f"KNN on 102 features                  : {plain:.4f}")
for n_comp in (2, 5, 10, 20):
    acc = cross_val_score(make_pipeline(StandardScaler(), PCA(n_components=n_comp),
                                        KNeighborsClassifier(15)),
                          X_wide, y_pca, cv=SKF).mean()
    print(f"KNN after PCA to {n_comp:>3} components        : {acc:.4f}")
print("\nPCA is not magic -- it keeps the directions of highest VARIANCE, which are not")
print("necessarily the informative ones. Here the noise has the same variance as the")
print("signal, so PCA only partly helps. Feature selection using the target does better.")

from sklearn.feature_selection import SelectKBest, f_classif
sel = cross_val_score(make_pipeline(StandardScaler(), SelectKBest(f_classif, k=5),
                                    KNeighborsClassifier(15)),
                      X_wide, y_pca, cv=SKF).mean()
print(f"\nKNN after supervised SelectKBest(5)  : {sel:.4f}")
print("(note: SelectKBest sits inside the pipeline, so the selection is refitted per fold")
print(" -- doing it outside would leak, as in statistics Notebook 10)")

---
## 4.7 Computational cost

| | Training | Prediction (one point) | Memory |
|---|---|---|---|
| Brute force | $O(1)$ | $O(nd)$ | $O(nd)$ |
| KD-tree | $O(dn\log n)$ | $O(\log n)$ if $d$ is small | $O(nd)$ |
| Ball tree | $O(dn\log n)$ | better than KD-tree for larger $d$ | $O(nd)$ |

`algorithm="auto"` picks for you: a KD-tree in low dimensions, brute force in high ones
(because above roughly 20 dimensions the trees stop helping — the curse again).

The real cost is that **the entire training set must be kept and searched at inference time**.
For a million rows and a latency budget of a few milliseconds, KNN is usually the wrong choice
unless you use an approximate nearest-neighbour index (FAISS, HNSW, ScaNN).

In [ ]:
import time

print(f"{'n_train':>9}{'d':>5}{'fit (ms)':>11}{'predict 1k (ms)':>18}")
for n_, d_ in [(1_000, 5), (10_000, 5), (100_000, 5), (10_000, 50), (10_000, 500)]:
    Xbig = rng.normal(size=(n_, d_))
    ybig = rng.integers(0, 2, n_)
    Xq = rng.normal(size=(1_000, d_))
    mdl = KNeighborsClassifier(10)
    t0 = time.perf_counter(); mdl.fit(Xbig, ybig); t1 = time.perf_counter()
    mdl.predict(Xq); t2 = time.perf_counter()
    print(f"{n_:>9,}{d_:>5}{(t1-t0)*1000:>11.1f}{(t2-t1)*1000:>18.1f}")
print("\nFit is nearly free (it just indexes). Prediction cost grows with BOTH n and d.")
print("Contrast with logistic regression, where prediction is a single dot product")
print("regardless of how much data you trained on.")

---
## 4.8 Practical issues

**Imbalanced classes.** The majority class wins votes simply by being more numerous. Fixes:
use `weights="distance"`, use a smaller $k$, resample, or threshold `predict_proba` rather
than using `predict`.

**Categorical features.** Euclidean distance on one-hot columns is workable (the distance
between two different categories is $\sqrt{2}$, between the same category 0), but mixing them
with scaled numeric features requires care about relative weighting. For mostly-categorical
data, use a **Gower distance** or a tree model instead.

**Missing values.** KNN cannot handle them — impute first. Amusingly, `KNNImputer` uses KNN to
do exactly that.

**Duplicate rows.** If duplicates straddle a train/test split, $k=1$ will look perfect. Always
deduplicate before splitting.

In [ ]:
# Imbalance: what k does to recall on the minority class
from sklearn.metrics import recall_score, precision_score, roc_auc_score
m_i = 2_000
Xi = rng.normal(size=(m_i, 4))
zi = -2.9 + 1.3*Xi[:, 0] + 0.9*Xi[:, 1]
yi = (rng.random(m_i) < 1/(1+np.exp(-zi))).astype(int)
Xa, Xb, ya, yb = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
print(f"Positive rate: {yi.mean():.4f}\n")

print(f"{'k':>5}{'weights':>10}{'accuracy':>10}{'precision':>11}{'recall':>9}{'ROC-AUC':>10}")
for k in (1, 5, 15, 50):
    for w in ("uniform", "distance"):
        mdl = make_pipeline(StandardScaler(), KNeighborsClassifier(k, weights=w)).fit(Xa, ya)
        pr = mdl.predict(Xb); pb = mdl.predict_proba(Xb)[:, 1]
        print(f"{k:>5}{w:>10}{accuracy_score(yb, pr):>10.4f}"
              f"{precision_score(yb, pr, zero_division=0):>11.4f}"
              f"{recall_score(yb, pr, zero_division=0):>9.4f}{roc_auc_score(yb, pb):>10.4f}")
print("\nAs k grows, recall on the rare class collapses -- with k=50 a minority point needs")
print("26 minority neighbours to win the vote, and there are rarely that many nearby.")
print("Note ROC-AUC still improves: the RANKING is fine, only the 0.5 threshold is wrong.")
print("So threshold the probabilities instead of using predict().")

In [ ]:
# Handling mixed types: one-hot plus scaling, inside a ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

m_x = 900
mixed = pd.DataFrame({
    "age": rng.uniform(18, 70, m_x),
    "spend": rng.lognormal(7, 0.6, m_x),
    "city": rng.choice(["A", "B", "C"], m_x),
    "plan": rng.choice(["basic", "pro"], m_x, p=[0.7, 0.3]),
})
mixed["target"] = ((mixed.age < 40).astype(int) + (mixed.plan == "pro").astype(int)
                   + (rng.random(m_x) < 0.3).astype(int) >= 2).astype(int)

prep = ColumnTransformer([
    ("num", StandardScaler(), ["age", "spend"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["city", "plan"]),
])
pipe_mix = make_pipeline(prep, KNeighborsClassifier(15))
print(f"KNN on mixed types, CV accuracy: "
      f"{cross_val_score(pipe_mix, mixed.drop(columns='target'), mixed.target, cv=SKF).mean():.4f}")
print("\nA caveat worth stating: after one-hot encoding, a 3-category feature contributes")
print("3 columns to the distance while a numeric feature contributes 1. The encoding")
print("silently changes the relative weight of your features -- something to check when")
print("categorical columns dominate.")

---
## 4.9 When to use KNN

**Good fit**

- Low-dimensional data (say up to ~20 informative features)
- Genuinely **local** structure — nearby points really do behave alike
- Irregular, non-linear decision boundaries
- Small to medium $n$, and no hard latency requirement
- As a **baseline**: it is two lines of code and tells you whether local structure exists
- **Recommendation and retrieval**, where "find similar items" *is* the task

**Poor fit**

- Many features, especially with irrelevant ones
- Large $n$ with tight latency budgets
- Need for interpretability or feature effects
- Missing values or heavily imbalanced classes
- Any need to **extrapolate**

**Where KNN quietly dominates:** modern embedding search. Every semantic search engine and
RAG pipeline is a nearest-neighbour lookup — just over learned embeddings, with an approximate
index, at a scale brute force could never handle.

In [ ]:
# NearestNeighbors for retrieval rather than prediction -- the 'find similar' use case
items = pd.DataFrame({
    "name": ["basic phone", "flagship phone", "budget laptop", "gaming laptop",
             "wireless earbuds", "studio headphones", "smart watch", "fitness band"],
    "price": [8_000, 90_000, 35_000, 160_000, 4_000, 22_000, 28_000, 3_500],
    "battery_hrs": [30, 20, 8, 4, 6, 35, 18, 120],
    "weight_g": [180, 210, 1_800, 2_600, 55, 320, 45, 25],
})
feat_cols = ["price", "battery_hrs", "weight_g"]
Xi_ = StandardScaler().fit_transform(items[feat_cols])

nn = NearestNeighbors(n_neighbors=3, metric="euclidean").fit(Xi_)
dist, idx = nn.kneighbors(Xi_)

print("Most similar items (excluding itself):")
for i, name in enumerate(items.name):
    similar = [f"{items.name[j]} (d={dist[i][r]:.2f})" for r, j in enumerate(idx[i]) if j != i]
    print(f"  {name:<20} -> {', '.join(similar)}")
print("\nNo labels, no training, no prediction -- just a similarity index. This is the")
print("mechanism behind content-based recommenders (Notebook 11).")

---
## Exercises

**Exercise 1.** On the iris dataset, find the best combination of $k$, weighting and distance
metric by grid search. Report the CV and test accuracy, and show the confusion matrix. Then
show what happens without scaling.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_iris

iris = load_iris()
Xi2, yi2 = iris.data, iris.target
Xa2, Xb2, ya2, yb2 = train_test_split(Xi2, yi2, test_size=0.3, random_state=0, stratify=yi2)

gs1 = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                   {"kneighborsclassifier__n_neighbors": range(1, 31, 2),
                    "kneighborsclassifier__weights": ["uniform", "distance"],
                    "kneighborsclassifier__metric": ["euclidean", "manhattan", "chebyshev"]},
                   cv=SKF, scoring="accuracy").fit(Xa2, ya2)

print(f"Best parameters : {gs1.best_params_}")
print(f"Best CV accuracy: {gs1.best_score_:.4f}")
print(f"Test accuracy   : {gs1.score(Xb2, yb2):.4f}\n")
print(pd.DataFrame(confusion_matrix(yb2, gs1.predict(Xb2)),
                   index=[f"true {n}" for n in iris.target_names],
                   columns=[f"pred {n}" for n in iris.target_names]))
print()
print(classification_report(yb2, gs1.predict(Xb2), target_names=iris.target_names))

no_scale = GridSearchCV(KNeighborsClassifier(),
                        {"n_neighbors": range(1, 31, 2)},
                        cv=SKF, scoring="accuracy").fit(Xa2, ya2)
print(f"Without scaling : CV {no_scale.best_score_:.4f}, test {no_scale.score(Xb2, yb2):.4f}")
print("Iris features are all in centimetres with similar ranges, so scaling barely matters")
print("here. That is the exception -- check the ranges before you assume it.")
print(f"\nFeature ranges: {np.round(Xi2.max(axis=0) - Xi2.min(axis=0), 2).tolist()}")

**Exercise 2.** Demonstrate the curse of dimensionality on real data. Take the digits dataset
(64 pixel features) and compare KNN accuracy at the full dimension against PCA-reduced
versions. Explain the shape of the result.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from sklearn.datasets import load_digits

dg = load_digits()
Xg, yg = dg.data, dg.target
print(f"{Xg.shape[0]} images, {Xg.shape[1]} pixel features, 10 classes\n")

results = []
for n_comp in [2, 5, 10, 20, 30, 40, 64]:
    pipe = make_pipeline(StandardScaler(), PCA(n_components=n_comp),
                         KNeighborsClassifier(5))
    acc = cross_val_score(pipe, Xg, yg, cv=SKF).mean()
    var = PCA(n_components=n_comp).fit(StandardScaler().fit_transform(Xg)
                                      ).explained_variance_ratio_.sum()
    results.append({"components": n_comp, "CV_accuracy": acc, "variance_kept": var})
full = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
                       Xg, yg, cv=SKF).mean()
print(pd.DataFrame(results).round(4).to_string(index=False))
print(f"\nAll 64 raw features, no PCA: {full:.4f}")
print()
print("Explanation of the shape:")
print("  * 2-5 components lose too much information -> accuracy suffers (underfitting)")
print("  * around 20-30 components is the sweet spot: the digit structure is retained and")
print("    the distance metric becomes better behaved")
print("  * the full 64 dimensions are not catastrophic here, because pixel features are")
print("    highly CORRELATED -- the intrinsic dimension is far below 64. The curse bites")
print("    on INDEPENDENT irrelevant features, which is why the synthetic experiment in")
print("    section 4.6 looked so much worse than this one.")

**Exercise 3.** Build a KNN model for an imbalanced problem (5% positives) and show that
tuning the probability threshold beats tuning $k$ alone. Report precision, recall and the
best achievable F1 for each approach.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
from sklearn.metrics import f1_score, precision_recall_curve

m_e = 4_000
Xe = rng.normal(size=(m_e, 5))
ze = -3.5 + 1.4*Xe[:, 0] + 1.0*Xe[:, 1] - 0.8*Xe[:, 2]
ye = (rng.random(m_e) < 1/(1+np.exp(-ze))).astype(int)
Xa3, Xb3, ya3, yb3 = train_test_split(Xe, ye, test_size=0.3, random_state=0, stratify=ye)
print(f"Positive rate: {ye.mean():.4f}\n")

print("Approach A -- tune k, keep the default 0.5 threshold:")
best_a = None
for k in (5, 15, 25, 51, 101):
    mdl = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(k, weights="distance")).fit(Xa3, ya3)
    pr = mdl.predict(Xb3)
    f1 = f1_score(yb3, pr, zero_division=0)
    print(f"  k={k:>4}: precision {precision_score(yb3, pr, zero_division=0):.4f}, "
          f"recall {recall_score(yb3, pr, zero_division=0):.4f}, F1 {f1:.4f}")
    if best_a is None or f1 > best_a[1]:
        best_a = (k, f1)

print(f"\nApproach B -- fix k, tune the threshold on predict_proba:")
mdl_b = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(51, weights="distance")).fit(Xa3, ya3)
pb3 = mdl_b.predict_proba(Xb3)[:, 1]
prec, rec, ths = precision_recall_curve(yb3, pb3)
f1s = 2*prec*rec / np.clip(prec + rec, 1e-12, None)
best_i = int(np.nanargmax(f1s[:-1]))
print(f"  best threshold {ths[best_i]:.4f}: precision {prec[best_i]:.4f}, "
      f"recall {rec[best_i]:.4f}, F1 {f1s[best_i]:.4f}")

print(f"\nBest F1 by tuning k alone      : {best_a[1]:.4f} (k={best_a[0]})")
print(f"Best F1 by tuning the threshold: {f1s[best_i]:.4f}")
print("\nThe threshold is the more powerful lever, and it is free -- one model, one")
print("sweep over predict_proba, no refitting. Tune k for ranking quality (ROC-AUC),")
print("then tune the threshold for the operating point you actually need.")
print("\nNote also that with k=51 and weights='uniform', predict_proba only takes 52")
print("distinct values, so the threshold sweep is coarse. Distance weighting gives a")
print("continuous score and therefore finer control -- a good reason to prefer it here.")

**Exercise 4 (challenge).** You are asked to build a "customers like this one" feature for an
e-commerce site: given a customer, return the 5 most similar others. Design it, decide on the
metric and the scaling, and explain how you would evaluate something with no labels.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
m_c = 2_000
customers = pd.DataFrame({
    "customer_id": np.arange(m_c),
    "orders_12m": rng.poisson(6, m_c),
    "avg_basket": rng.lognormal(6.4, 0.7, m_c),
    "days_since_last": rng.exponential(40, m_c).clip(0, 400),
    "categories_bought": rng.integers(1, 9, m_c),
    "returns_rate": rng.beta(1.5, 12, m_c),
})
behaviour_cols = ["orders_12m", "avg_basket", "days_since_last",
                  "categories_bought", "returns_rate"]

print("DESIGN DECISIONS\n")
print("1. Metric. Euclidean on standardised features, because all five features are")
print("   magnitudes we care about directly (a customer with twice the basket size IS")
print("   different). Cosine would be right if these were sparse counts across hundreds")
print("   of product categories, where only the PATTERN matters, not the volume.")
print("2. Scaling. RobustScaler, not StandardScaler: avg_basket and days_since_last are")
print("   heavily right-skewed, so means and standard deviations are dragged by the tail.")
print("3. Skewed features get a log first, so that 'twice as much' is a constant distance.")
print()

from sklearn.preprocessing import RobustScaler

work = customers.copy()
work["avg_basket"] = np.log1p(work.avg_basket)
work["days_since_last"] = np.log1p(work.days_since_last)
Z = RobustScaler().fit_transform(work[behaviour_cols])

index = NearestNeighbors(n_neighbors=6, metric="euclidean").fit(Z)
dist_c, idx_c = index.kneighbors(Z)

def similar_to(cid, n=5):
    row = np.where(customers.customer_id == cid)[0][0]
    neigh = [j for j in idx_c[row] if j != row][:n]
    out = customers.iloc[[row] + neigh].copy()
    out.insert(1, "role", ["QUERY"] + [f"match {i+1}" for i in range(len(neigh))])
    out.insert(2, "distance", [0.0] + [dist_c[row][list(idx_c[row]).index(j)] for j in neigh])
    return out.round(3)

print("Example output for customer 42:")
print(similar_to(42).to_string(index=False))

In [ ]:
print("\nEVALUATION WITHOUT LABELS\n")

# (a) Internal consistency: are neighbours more similar than random pairs on a HELD-OUT
#     behaviour we did not index on?
customers["next_month_spend"] = (
    120 * customers.orders_12m + 0.4*customers.avg_basket
    - 3*customers.days_since_last + rng.normal(0, 400, m_c)).clip(0, None)

neigh_gap, random_gap = [], []
for row in range(0, m_c, 3):
    neigh = [j for j in idx_c[row] if j != row][:5]
    rand = rng.choice(m_c, 5, replace=False)
    own = customers.next_month_spend[row]
    neigh_gap.append(np.abs(customers.next_month_spend.iloc[neigh] - own).mean())
    random_gap.append(np.abs(customers.next_month_spend.iloc[rand] - own).mean())

print("(a) HOLD-OUT BEHAVIOUR TEST -- the key quantitative check.")
print(f"    Mean |difference| in next-month spend vs 5 nearest neighbours: "
      f"{np.mean(neigh_gap):>8,.0f}")
print(f"    Mean |difference| vs 5 random customers                     : "
      f"{np.mean(random_gap):>8,.0f}")
print(f"    Ratio: {np.mean(neigh_gap)/np.mean(random_gap):.3f} "
      f"(lower is better; 1.0 would mean the index is useless)")
print("    Interpretation: neighbours predict a behaviour that was NOT part of the")
print("    distance calculation, so the notion of similarity generalises.\n")

print("(b) STABILITY. Re-fit on a 90% subsample and measure overlap of the top-5 lists.")
sub = rng.choice(m_c, int(0.9*m_c), replace=False)
Z_sub = Z[sub]
idx_map = {orig: i for i, orig in enumerate(sub)}
nn_sub = NearestNeighbors(n_neighbors=6).fit(Z_sub)
_, idx_s = nn_sub.kneighbors(Z_sub)
overlaps = []
for orig in sub[:400]:
    r_full = [j for j in idx_c[orig] if j != orig][:5]
    r_sub = [sub[j] for j in idx_s[idx_map[orig]] if sub[j] != orig][:5]
    overlaps.append(len(set(r_full) & set(r_sub)) / 5)
print(f"    Mean top-5 overlap after dropping 10% of customers: {np.mean(overlaps):.3f}")
print("    A stable index should stay well above ~0.6 here.\n")

print("(c) ONLINE EVALUATION -- the only test that settles it.")
print("    A/B test the feature: does a 'customers like you also bought' module lift")
print("    click-through and conversion against the current recommender? Measure per")
print("    the statistics module: fixed sample size from a power calculation, one look,")
print("    effect size and confidence interval reported alongside the p-value.\n")

print("(d) OPERATIONAL CHECKS to run before launch:")
print("    * cold start: new customers have no history -- fall back to popularity")
print("    * privacy: never surface another customer's identity or raw attributes")
print("    * cost: 2,000 rows is trivial, but 20 million needs an approximate index")
print("      (HNSW/FAISS) and a nightly rebuild")
print("    * drift: re-fit the scaler and index on a schedule; behaviour distributions move")

---
## Summary

| Concept | Key point |
|---|---|
| Algorithm | Find the $k$ nearest training points; vote or average |
| Lazy learning | No training; all cost at prediction time |
| Metric | Euclidean (default), Manhattan, Minkowski, **cosine** for text |
| **Scaling** | **Mandatory** — otherwise the largest-range feature is the only feature |
| $k$ | Capacity **decreases** as $k$ grows; $k=1$ has zero training error |
| Odd $k$ | Avoids ties in binary classification |
| `weights="distance"` | Closer neighbours count more; makes large $k$ usable |
| Regression | Average of neighbours; piecewise constant; **cannot extrapolate** |
| Curse of dimensionality | Distances concentrate; irrelevant features are fatal |
| Remedies | Feature selection, PCA, or a different model |
| Cost | $O(nd)$ per prediction; use approximate indexes at scale |
| Imbalance | Large $k$ destroys minority recall — threshold the probabilities instead |
| Best uses | Low dimension, local structure, retrieval and recommendation |

**Next up:** [Notebook 5 — Clustering](5.%20Clustering.ipynb), our first **unsupervised**
method: finding groups when nobody gave you labels.